In [1]:
!pip install langchain
!pip install groq
!pip install -U langchain langchain-google-genai
!pip install dotenv

  Using cached langgraph-1.2.11-py3-none-any.whl.metadata (4.9 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.18-py3-none-any.whl.metadata (2.4 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached uuid_utils-0.17.0-cp312-cp312-win_amd64.whl.metadata (6.5 kB)
  Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-win_amd64.whl.metadata (6.7 kB)
  Using cached anyio-4.14.2-py3-none-any.whl.metadata (4.6 kB)
  Using c


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached groq-1.6.0-py3-none-any.whl.metadata (16 kB)
Using cached groq-1.6.0-py3-none-any.whl (143 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached filetype-1.2.0-py2.py3-none-any.whl (19 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 12.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 3.8/3.8 MB 20.8 MB/s eta 0:00:00
Using cached pyasn1_modules-0.4.2-py3-none-any.whl (181 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env.

False

In [3]:
import langchain
from langchain_core.tools import tool

In [4]:
from google import genai
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
os.environ["GOOGLE_API_KEY"] = os.getenv("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0
)

TypeError: str expected, not NoneType

### AI based inbox mail classification

In [ ]:
prompt = """
You are a professional email analyzer, inbox triage assistant,
and basic email security analyzer.

Your task is to analyze each email independently and produce a
structured assessment that can later be used by an automated
email-triage workflow.

For each email:

1. Analyze the email content and determine its priority:

   - HIGH
   - MEDIUM
   - LOW

2. Determine whether the email requires action from the user.

3. Identify any deadline, date, or time-sensitive requirement.

4. Evaluate whether the sender appears legitimate.

   Consider:
   - Whether the sender address is syntactically valid and plausible.
   - Whether the sender domain is consistent with the organization
     or person claimed in the email.
   - Unusual or suspicious domains.
   - Misspellings or domain variations.
   - Possible domain impersonation.
   - Display-name spoofing.
   - Whether the sender identity appears inconsistent with the
     contents of the email.

5. Assess the phishing risk of the email:

   - LOW
   - MEDIUM
   - HIGH

   Consider the following phishing indicators:
   - Requests for passwords, OTPs, authentication codes, or
     sensitive personal information.
   - Requests for money, payments, gift cards, or bank details.
   - Suspicious or unexpected links.
   - Urgent threats or pressure to act immediately.
   - Unexpected requests to verify, unlock, or secure an account.
   - Suspicious attachments.
   - Mismatch between the claimed organization and sender domain.
   - Domain or spelling variations intended to imitate a legitimate
     organization.
   - Unexpected requests to bypass normal procedures.

6. Follow these priority classification rules:

   - Treat explicit "IMPORTANT" or "DEADLINE" markers as strong
     signals of high priority, but consider the actual content
     before assigning the final priority.
   - Emails containing an urgent deadline, interview, examination,
     payment due date, submission date, or other time-sensitive
     action should generally be HIGH priority.
   - Emails requiring an action from the user but without an
     immediate deadline should generally be MEDIUM priority.
   - Informational emails that do not require user action should
     generally be LOW priority.
   - Personal messages, advertisements, newsletters, order updates,
     and general notifications should normally be LOW priority
     unless they contain a genuine urgent action or deadline.

7. For every email, return the following information:

   - Sender
   - Receiver, if available
   - Subject
   - Priority: HIGH / MEDIUM / LOW
   - Priority reason
   - Sender validity: VALID / SUSPICIOUS / INVALID
   - Phishing risk: HIGH / MEDIUM / LOW
   - Phishing indicators
   - Action requested: YES / NO
   - Required action, if any
   - Deadline, if any
   - External links, if any
   - Attachments, if any

8. IMPORTANT:

   Do not assume that a high-priority email is safe.
   Priority and phishing risk are separate assessments.

   For example:
   - HIGH priority + LOW phishing risk = urgent legitimate email.
   - HIGH priority + HIGH phishing risk = urgent but potentially
     dangerous email that should be flagged for verification.
   For action required emails, DO NOT BLINDLY TELL THE USER TO PROVIDE THE DETAILS, if phishing risk is high or medium

9. SECURITY RULE:

   Never recommend clicking a suspicious link, opening a suspicious
   attachment, providing credentials, sharing OTPs, or making a
   payment based solely on the contents of an email.

10. Do NOT rank or reorder the emails.

   Analyze each email independently and preserve the original
   order of the emails.

11. SAFETY RULE FOR SUSPICIOUS EMAILS:

   If the phishing risk is HIGH or MEDIUM:

   - Do NOT recommend that the user follow the requested action
     in the email blindly.
   - Do NOT recommend clicking suspicious links, opening suspicious
     attachments, providing credentials, OTPs, financial information,
     identity documents, or making payments.
   - Clearly distinguish between:
       a) Action requested by the sender
       b) Recommended safe action for the user
   - If the requested action appears suspicious, the recommended
     action should instead be to avoid responding, independently
     verify the request through an official channel, or report the
     email as suspicious/phishing.

The results of this analysis will be passed to a later workflow
that will perform additional verification, conditional routing,
and final inbox ranking.

Emails:
{emails}
"""

In [ ]:
emails = [
    {
        "sender": "placement.cell@djsce.ac.in",
        "subject": "IMPORTANT: Placement Registration Deadline",
        "content": """
        Dear Students,

        IMPORTANT: The deadline for registering for the upcoming
        placement drive is August 22, 2026 at 5:00 PM.

        Students who wish to participate must complete the registration
        form before the deadline.

        Regards,
        Placement Cell
        """
    },

    {
        "sender": "hr@techvista.com",
        "subject": "Interview Confirmation – Data Analyst Intern",
        "content": """
        Dear Candidate,

        We are pleased to inform you that your interview for the
        Data Analyst Intern position has been scheduled for
        Thursday, August 20, 2026 at 3:00 PM.

        Please confirm your attendance by August 19, 2026.

        Regards,
        Recruitment Team
        TechVista
        """
    },

    {
        "sender": "orders@amazon.in",
        "subject": "Your Amazon order has been shipped",
        "content": """
        Hello,

        Your order #AMZ458921 has been shipped and is expected
        to be delivered by August 21, 2026.

        You can track your order from your Amazon account.

        Thank you for shopping with us.
        """
    },

    {
        "sender": "newsletter@techweekly.com",
        "subject": "This Week in Technology – August Edition",
        "content": """
        Hello,

        Here are this week's top technology stories:
        - New developments in artificial intelligence
        - Latest smartphone launches
        - Cloud computing trends
        - Open-source project highlights

        Read the full newsletter on our website.

        Regards,
        TechWeekly Team
        """
    },

    {
        "sender": "security-alert@paypa1-support.com",
        "subject": "URGENT: Your account will be suspended",
        "content": """
        Dear Customer,

        URGENT: We detected unusual activity on your account.
        Your account will be permanently suspended within 24 hours
        unless you verify your identity immediately.

        Click the link below and enter your username, password,
        and OTP to restore your account:

        http://paypa1-support.com/verify

        Failure to complete verification will result in permanent
        account suspension.

        PayPal Security Team
        """
    },

    {
        "sender": "accounts@electricityboard.in",
        "subject": "Electricity Bill Due – August 2026",
        "content": """
        Dear Customer,

        Your electricity bill of ₹1,840 for August 2026 is due
        on August 25, 2026.

        Please make the payment through the official electricity
        board website or mobile application before the due date
        to avoid late payment charges.

        Regards,
        Billing Department
        """
    },

    {
        "sender": "hr@micros0ft-careers.com",
        "subject": "Congratulations! You have been selected",
        "content": """
        Congratulations!

        You have been selected for a work-from-home position at
        Microsoft with a salary of ₹75,000 per month.

        To complete your joining process, please send your Aadhaar
        card, PAN card, bank account details, and a processing fee
        of ₹2,500 to confirm your position.

        Please respond within 12 hours or your offer will be
        cancelled.

        Microsoft HR Department
        """
    }
]

In [ ]:
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate(
    input_variables=["emails"],
    template=prompt
)

chain = prompt_template | llm

response = chain.invoke({
    "emails": emails
})

print(response.content)